# Seeded Random Quantum Circuits: 1–8 Qubits

**Qiskit | Controlled random circuit generation | Ideal, shot-based simulation**

This notebook uses Qiskit's `random_circuit` function to generate eight reproducible circuits with 1–8 qubits, then evaluates them with `StatevectorSampler`. For each size, it prints circuit depth and gate counts, shows a circuit diagram and plots the sampled measurement outcomes.

**Reproducibility:** The circuit-generation seeds are 101, 202, …, 808. The sampler uses the same integer seed for that circuit. The shot budget is **4,096** per circuit. Circuit-generation depth and gate arity follow the source notebooks.

**Scientific scope:** `random_circuit` randomly chooses circuit operations within the supplied configuration; it does not guarantee a uniform output distribution, a specific entanglement structure, hardware-native gates or any performance improvement. Sampling is **ideal/noiseless** apart from finite-shot sampling variability. No quantum hardware or error correction is involved.

**Setup:** Run `pip install -r requirements.txt` from the repository root, or install `qiskit[visualization]==2.5.2` and `matplotlib` in Colab. Then run all cells.

In [ ]:
from qiskit.circuit.random import random_circuit
from qiskit.primitives import StatevectorSampler
from qiskit.visualization import plot_histogram
from IPython.display import display

SHOTS = 4096
DEPTHS = {1: 10, 2: 10, 3: 10, 4: 12, 5: 12, 6: 12, 7: 14, 8: 14}


def run_random_circuit_experiment(num_qubits: int):
    """Generate and ideally sample a reproducible circuit with 1–8 qubits."""
    if not isinstance(num_qubits, int) or num_qubits not in DEPTHS:
        raise ValueError("num_qubits must be an integer from 1 to 8")
    seed = num_qubits * 101
    circuit = random_circuit(
        num_qubits=num_qubits,
        depth=DEPTHS[num_qubits],
        max_operands=min(4, num_qubits),
        measure=True,
        seed=seed,
    )
    sampler = StatevectorSampler(seed=seed)
    result = sampler.run([circuit], shots=SHOTS).result()[0]
    # random_circuit(..., measure=True) currently names its classical register 'c'.
    # Looking up its actual name avoids relying on a hard-coded register label.
    register_name = circuit.cregs[0].name
    counts = getattr(result.data, register_name).get_counts()

    assert circuit.num_qubits == num_qubits
    assert sum(counts.values()) == SHOTS
    assert all(len(bitstring) == num_qubits for bitstring in counts)

    print(f"{num_qubits}-qubit seeded random circuit | seed={seed} | shots={SHOTS}")
    print("Requested generation depth:", DEPTHS[num_qubits])
    print("Actual circuit depth (includes measurements):", circuit.depth())
    print("Operations:", dict(circuit.count_ops()))
    print("Measurement counts:", counts)
    display(circuit.draw(output='mpl', style='iqp', fold=100))
    display(plot_histogram(
        counts,
        number_to_keep=16 if num_qubits >= 6 else None,
        title=f"{num_qubits}-qubit random circuit: ideal samples ({SHOTS} shots)",
    ))
    return circuit, counts


## 1 qubit · seeded random circuit

Generator seed: **101** · requested circuit depth: **10** · maximum gate operands: **1**.

In [ ]:
random_circuit_1, random_counts_1 = run_random_circuit_experiment(1)

## 2 qubits · seeded random circuit

Generator seed: **202** · requested circuit depth: **10** · maximum gate operands: **2**.

In [ ]:
random_circuit_2, random_counts_2 = run_random_circuit_experiment(2)

## 3 qubits · seeded random circuit

Generator seed: **303** · requested circuit depth: **10** · maximum gate operands: **3**.

In [ ]:
random_circuit_3, random_counts_3 = run_random_circuit_experiment(3)

## 4 qubits · seeded random circuit

Generator seed: **404** · requested circuit depth: **12** · maximum gate operands: **4**.

In [ ]:
random_circuit_4, random_counts_4 = run_random_circuit_experiment(4)

## 5 qubits · seeded random circuit

Generator seed: **505** · requested circuit depth: **12** · maximum gate operands: **4**.

In [ ]:
random_circuit_5, random_counts_5 = run_random_circuit_experiment(5)

## 6 qubits · seeded random circuit

Generator seed: **606** · requested circuit depth: **12** · maximum gate operands: **4**.

In [ ]:
random_circuit_6, random_counts_6 = run_random_circuit_experiment(6)

## 7 qubits · seeded random circuit

Generator seed: **707** · requested circuit depth: **14** · maximum gate operands: **4**.

In [ ]:
random_circuit_7, random_counts_7 = run_random_circuit_experiment(7)

## 8 qubits · seeded random circuit

Generator seed: **808** · requested circuit depth: **14** · maximum gate operands: **4**.

In [ ]:
random_circuit_8, random_counts_8 = run_random_circuit_experiment(8)

## Interpretation and limitations

Each histogram is one shot-sampled realization of an ideal statevector-derived measurement distribution. A random circuit can yield highly concentrated outcomes (including `00…0`), so observing a narrow histogram does **not** by itself indicate a broken implementation. The random gates are not restricted to the native gate set of a particular quantum processor.

To study noise, correlated errors, readout error, quantum error mitigation or QEC, a separate physical noise model or real-device execution workflow is necessary.